# InterPro Query

In [ ]:
# %pip install biopython

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
import gzip
import re
from Bio import SeqIO

In [ ]:
gtf_path = './Raw/Iso-Caller/iso_ref/Mus_musculus.GRCm39.109.sorted.gtf'

fasta_path = './Raw/Iso-Caller/iso_ref/Mus_musculus.GRCm39.pep.all.fa.gz' 

isoforms = [
    "Il16-201", "Il16-203", "Il1b-201", "Il1b-203", "Il1rn-201", "Il1rn-204",
    "Tgfb2-201", "Tgfb2-206", "Tgfbr1-201", "Tgfbr1-202", "Tgfbr2-201", "Tgfbr2-202",
    "Csf1-201", "Csf1-203", "Csf1-206", "Csf1r-201", "Csf1r-202", "Csf2ra-201",
    "Csf2ra-202", "Csf2ra-207", "Csf2rb-201", "Csf2rb-206", "Csf2rb2-201", "Csf2rb2-202",
    "Csf3r-201", "Csf3r-202", "Ifnar1-201", "Ifnar1-202", "Ifnar1-203", "Ifngr1-201",
    "Ifngr1-203", "Il10ra-201", "Il10ra-202", "Il17ra-201", "Il17ra-203", "Il4ra-201",
    "Il4ra-205", "Il6ra-201", "Il6ra-202", "Il6st-201", "Il6st-203", "Il7r-201",
    "Il7r-204", "Tnfaip2-201", "Tnfaip2-202", "Tnfaip2-207", "Tnfaip3-201",
    "Tnfaip3-202", "Tnfaip3-205", "Tnfrsf12a-201", "Tnfrsf12a-202", "Tnfrsf12a-206",
    "Tnfrsf9-201", "Tnfrsf9-203", "Tnfrsf9-204", "Tnfrsf9-205"
]

output_path = './extracted_protein_isoforms.fasta'

In [ ]:
def extract_protein_sequences():
    # Parse GTF to map common names to Ensembl Transcript IDs
    print("Parsing GTF to map isoform names to Ensembl IDs...")
    name_to_id = {}
    
    with open(gtf_path, 'r') as gtf:
        for line in gtf:
            if line.startswith('#'): continue
            
            if '\ttranscript\t' in line or '\texon\t' in line:
                t_name_match = re.search(r'transcript_name "([^"]+)"', line)
                t_id_match = re.search(r'transcript_id "([^"]+)"', line)
                
                if t_name_match and t_id_match:
                    t_name = t_name_match.group(1)
                    t_id = t_id_match.group(1)
                    if t_name in isoforms:
                        name_to_id[t_name] = t_id

    id_to_name = {v: k for k, v in name_to_id.items()}
    target_ids = set(name_to_id.values())

    print(f"Mapped {len(target_ids)} out of {len(isoforms)} requested isoforms.")
    
    missing = set(isoforms) - set(name_to_id.keys())
    if missing:
        print(f"Warning: Could not find these isoforms in the GTF: {missing}")

    # Extract amino acid sequences from the compressed PEP FASTA file
    print("\nExtracting protein sequences from FASTA...")
    extracted_records = []
    
    with gzip.open(fasta_path, "rt") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            transcript_match = re.search(r'transcript:(ENSMUST\d+)', record.description)
            
            if transcript_match:
                transcript_id = transcript_match.group(1)
                
                if transcript_id in target_ids:
                    isoform_name = id_to_name[transcript_id]
                    
                    record.id = isoform_name
                    record.description = "" 
                    extracted_records.append(record)

    # Write to output
    print(f"\nWriting {len(extracted_records)} protein sequences to {output_path}...")
    SeqIO.write(extracted_records, output_path, "fasta")
    print("Extraction complete. ")

if __name__ == "__main__":
    extract_protein_sequences()

In [ ]:
import requests
import time
from Bio import SeqIO

# Configuration
input_fasta = './extracted_protein_isoforms.fasta'
output_tsv = './interpro_batch_results.tsv'
email = "mymail@gmail.com" # mandatory

base_url = "https://www.ebi.ac.uk/Tools/services/rest/iprscan5"

def submit_to_interpro(clean_fasta_string, clean_id):
    """Submits a sequence to InterProScan and returns the Job ID."""
    run_url = f"{base_url}/run"
    payload = {
        'email': email,
        'title': clean_id,
        'goterms': 'false', 
        'pathways': 'false',
        'sequence': clean_fasta_string
    }
    
    response = requests.post(run_url, data=payload)
    response.raise_for_status()
    return response.text

def check_status(job_id):
    """Polls the server to check if the job is finished."""
    status_url = f"{base_url}/status/{job_id}"
    while True:
        response = requests.get(status_url)
        status = response.text
        if status == "FINISHED":
            return True
        elif status in ["ERROR", "FAILED", "NOT_FOUND"]:
            print(f"Job {job_id} failed with status: {status}")
            return False
        
        # 15-second delay to respect EBI's API limits
        time.sleep(15) 

def get_results(job_id):
    """Downloads the TSV results for a finished job."""
    result_url = f"{base_url}/result/{job_id}/tsv"
    response = requests.get(result_url)
    response.raise_for_status()
    return response.text

def run_batch_interpro():
    print("Starting InterProScan Batch Analysis...")
    
    with open(output_tsv, 'w') as out_file:
        for record in SeqIO.parse(input_fasta, "fasta"):
            clean_id = record.id 
            
            clean_fasta_string = f">{clean_id}\n{str(record.seq)}"
            
            print(f"\nSubmitting {clean_id}...")
            try:
                job_id = submit_to_interpro(clean_fasta_string, clean_id)
                print(f"Job ID: {job_id}. Waiting for results...")
                
                if check_status(job_id):
                    tsv_data = get_results(job_id)
                    out_file.write(tsv_data)
                    print(f"Success! Saved domains for {clean_id}.")
                
                # Politeness delay between submissions
                time.sleep(5) 
                
            except Exception as e:
                print(f"An error occurred with {clean_id}: {e}")

if __name__ == "__main__":
    if email == "":
        print("Please update the 'email' variable in the script before running!")
    else:
        run_batch_interpro()

# Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import os
import re

# Isoform List
isoforms = [
    "Il16-201", "Il16-203", "Il1b-201", "Il1b-203", "Il1rn-201", "Il1rn-204",
    "Tgfb2-201", "Tgfb2-206", "Tgfbr1-201", "Tgfbr1-202", "Tgfbr2-201", "Tgfbr2-202",
    "Csf1-201", "Csf1-203", "Csf1-206", "Csf1r-201", "Csf1r-202", "Csf2ra-201",
    "Csf2ra-202", "Csf2ra-207", "Csf2rb-201", "Csf2rb-206", "Csf2rb2-201", "Csf2rb2-202",
    "Csf3r-201", "Csf3r-202", "Ifnar1-201", "Ifnar1-202", "Ifnar1-203", "Ifngr1-201",
    "Ifngr1-203", "Il10ra-201", "Il10ra-202", "Il17ra-201", "Il17ra-203", "Il4ra-201",
    "Il4ra-205", "Il6ra-201", "Il6ra-202", "Il6st-201", "Il6st-203", "Il7r-201",
    "Il7r-204", "Tnfaip2-201", "Tnfaip2-202", "Tnfaip2-207", "Tnfaip3-201",
    "Tnfaip3-202", "Tnfaip3-205", "Tnfrsf12a-201", "Tnfrsf12a-202", "Tnfrsf12a-206",
    "Tnfrsf9-201", "Tnfrsf9-203", "Tnfrsf9-204", "Tnfrsf9-205"
]

columns = ['Protein_Accession', 'MD5', 'Length', 'Analysis', 'Signature_Accession', 
           'Signature_Description', 'Start', 'Stop', 'Score', 'Status', 'Date', 
           'InterPro_Accession', 'InterPro_Description', 'GO', 'Pathways']

df = pd.read_csv('interpro_batch_results.tsv', sep='\t', names=columns, header=None)
df['Gene'] = df['Protein_Accession'].str.split('-').str[0]
for col in ['Length', 'Start', 'Stop']: df[col] = pd.to_numeric(df[col])

df['Raw_Domain'] = df['InterPro_Description'].replace('-', np.nan).fillna(df['Signature_Description'])

# COMPRESSION
def compress_domain(name):
    n = str(name).lower()
    
    # Routing & Anchors
    if 'signal' in n: return 'Signal Peptide (SP) Region'
    if 'transmembrane' in n or 'membrane' in n: return 'Transmembrane Anchor (TM) Region'
    if 'disorder' in n or 'cytoplasm' in n: return 'Intracellular Tail (IDR) Region'
    
    # Receptors & Structural Folds
    if 'fibronectin' in n: return 'Fibronectin Domain'
    if 'immunoglobulin' in n or 'igc' in n: return 'Immunoglobulin (Ig) Fold'
    if 'receptor' in n: return 'Receptor Architecture Domain'
    
    # Aggressive Family Compression (Strict Word Boundaries!)
    if re.search(r'\b(il-?16|interleukin-16)', n): return 'IL-16 Family Domain'
    if re.search(r'\b(il-?17|interleukin-17)', n): return 'IL-17 Family Domain'
    if re.search(r'\b(il-?10|interleukin-10)', n): return 'IL-10 Family Domain'
    
    if re.search(r'\b(il-?1\b|il-?1[a-z]|interleukin-1\b|beta-trefoil|il1_2)', n): return 'IL-1 Family Domain'
    
    if 'tgf' in n or 'transforming growth factor' in n: return 'TGF-beta Family Domain'
    if 'tnf' in n or 'tumor necrosis factor' in n or 'a20' in n: return 'TNF/TNFAIP Family Domain'
    if 'csf' in n or 'colony' in n: return 'Colony Stimulating Factor (CSF) Domain'
    if 'pdz' in n: return 'PDZ Domain'
    if 'otu' in n: return 'OTU Deubiquitinase Domain'
    
    # Kinases & Enzymes
    if any(k in n for k in ['kinase', 'transferase', 'ptkc', 'stkc', 'tyrpk']): return 'Kinase Domain'
    
    if name == '-': return np.nan
    return name 

# OVERWRITE MEMORY WITH NEW CLASSIFICATIONS
df['Compressed_Domain'] = df['Raw_Domain'].apply(compress_domain)
df = df.dropna(subset=['Compressed_Domain']) 

base_matrix = pd.crosstab(df['Protein_Accession'], df['Compressed_Domain'])
base_matrix = (base_matrix > 0).astype(int)

found_isoforms = base_matrix.index.tolist()
missing_isoforms = list(set(isoforms) - set(found_isoforms))

if missing_isoforms:
    zero_df = pd.DataFrame(0, index=missing_isoforms, columns=base_matrix.columns)
    base_matrix = pd.concat([base_matrix, zero_df])

base_matrix.to_csv('compressed_one_hot_domains.csv')
print(f"Matrix Compressed! Down to {len(base_matrix.columns)} unified domains. ({len(missing_isoforms)} drop-outs injected).")

Matrix Compressed! Down to 25 unified domains. (6 drop-outs injected).


In [2]:
os.makedirs('plots', exist_ok=True)
genes = df['Gene'].unique()

for plot_gene in genes:
    gene_df = df[df['Gene'] == plot_gene].copy()
    isoforms = sorted(gene_df['Protein_Accession'].unique(), reverse=True)
    
    fig, ax = plt.subplots(figsize=(12, max(3, len(isoforms) * 1.5)))
    
    unique_domains = gene_df['Compressed_Domain'].unique()
    cmap = plt.get_cmap('tab20')
    
    domain_colors = {}
    for i, dom in enumerate(unique_domains):
        if 'Signal Peptide' in dom: domain_colors[dom] = 'cyan'
        elif 'Transmembrane' in dom: domain_colors[dom] = 'red'
        elif 'Intracellular' in dom: domain_colors[dom] = 'lightgreen'
        else: domain_colors[dom] = cmap(i % 20)
    
    y_ticks, y_labels = [], []
    
    for i, isoform in enumerate(isoforms):
        iso_df = gene_df[gene_df['Protein_Accession'] == isoform].copy()
        length = int(iso_df['Length'].iloc[0])
        y_pos = i 
        y_ticks.append(y_pos)
        y_labels.append(f"{isoform}\n({length} aa)")
        
        ax.hlines(y_pos, 0, length, color='grey', linewidth=4, zorder=1)
        
        iso_df['Draw_Length'] = iso_df['Stop'] - iso_df['Start']
        iso_df = iso_df.sort_values(by='Draw_Length', ascending=False)
        
        for _, row in iso_df.iterrows():
            start, stop, domain = int(row['Start']), int(row['Stop']), row['Compressed_Domain']
            color = domain_colors[domain]
            
            rect = patches.Rectangle((start, y_pos - 0.35), stop - start, 0.7, 
                                     facecolor=color, edgecolor='black', linewidth=0.5, alpha=1.0, zorder=2)
            ax.add_patch(rect)

    legend_handles = [patches.Patch(color=color, label=dom) for dom, color in domain_colors.items()]
    
    ax.legend(handles=legend_handles, bbox_to_anchor=(1.05, 1), loc='upper left', title="All Domains")
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)
    ax.set_xlabel('Amino Acid Position')
    ax.set_title(f'Domain Architecture: {plot_gene}')
    ax.set_ylim(-1, len(isoforms))
    ax.margins(y=0.1)
    
    plt.tight_layout()
    plt.savefig(f'./Figures/Isoform Analysis/{plot_gene}_architecture.png', dpi=600, bbox_inches='tight')
    plt.close(fig)

print("All dynamic ribbon plots generate.")

All dynamic ribbon plots generate.


In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# BUILD THE DOMAIN COMPLETENESS MATRIX
df['Domain_Length'] = df['Stop'] - df['Start']
domain_coverage = df.groupby(['Protein_Accession', 'Compressed_Domain'])['Domain_Length'].sum().unstack(fill_value=0)

completeness_matrix = pd.DataFrame(0.0, index=domain_coverage.index, columns=domain_coverage.columns)
genes = list(set([iso.split('-')[0] for iso in completeness_matrix.index]))

for gene in genes:
    canon = f"{gene}-201"
    gene_isos = [iso for iso in completeness_matrix.index if iso.startswith(gene+'-')]
    
    if canon in domain_coverage.index:
        canon_profile = domain_coverage.loc[canon]
        for iso in gene_isos:
            iso_profile = domain_coverage.loc[iso]
            fraction = iso_profile / canon_profile.replace(0, np.nan)
            fraction = fraction.fillna(iso_profile.apply(lambda x: 1.0 if x > 0 else 0.0))
            fraction = fraction.clip(upper=1.0)
            completeness_matrix.loc[iso] = fraction

missing_isoforms = list(set(isoforms) - set(completeness_matrix.index))
if missing_isoforms:
    zero_df = pd.DataFrame(0.0, index=missing_isoforms, columns=completeness_matrix.columns)
    completeness_matrix = pd.concat([completeness_matrix, zero_df])

completeness_matrix.to_csv('domain_completeness_matrix.csv')

In [4]:
# TARGETED FRACTIONAL CONTRAST HEATMAP
target_matrix = completeness_matrix[completeness_matrix.sum(axis=1) > 0].copy()
contrast_isoforms = set() 

for gene in genes:
    canon = f"{gene}-201"
    gene_isos = [iso for iso in target_matrix.index if iso.startswith(gene+'-')]
    
    if canon in target_matrix.index:
        canon_profile = target_matrix.loc[canon]
        for iso in gene_isos:
            if iso == canon: continue
            iso_profile = target_matrix.loc[iso]
            
            if not np.allclose(canon_profile, iso_profile, atol=0.05):
                contrast_isoforms.add(canon)
                contrast_isoforms.add(iso)

if contrast_isoforms:
    # Sort the set alphabetically into a list before slicing
    sorted_isoforms = sorted(list(contrast_isoforms))
    filtered_decoy_matrix = target_matrix.loc[sorted_isoforms]
    
    # Drop empty columns
    filtered_decoy_matrix = filtered_decoy_matrix.loc[:, (filtered_decoy_matrix != 0).any(axis=0)]
    
    filtered_decoy_matrix = filtered_decoy_matrix.T
    
    n_rows, n_cols = filtered_decoy_matrix.shape
    
    if n_rows > 1 and n_cols > 1:
        plt.figure(figsize=(max(8, n_cols * 0.6), max(6, n_rows * 0.5)))
        g_target = sns.clustermap(
            filtered_decoy_matrix, 
            cmap="Blues", 
            vmin=0.0, vmax=1.0,
            yticklabels=True, 
            xticklabels=True,
            figsize=(max(8, n_cols * 0.6), max(6, n_rows * 0.5)),
            linewidths=0.5,
            linecolor='lightgrey',
            row_cluster=True,   
            col_cluster=False, 
            cbar_pos=(1.02, 0.4, 0.0225, 0.35) 
        )
        
        plt.setp(g_target.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=10)
        plt.setp(g_target.ax_heatmap.get_yticklabels(), rotation=0, fontsize=10)
        
        # Legend formatting
        g_target.cax.set_yticks([0.0, 0.5, 1.0])
        g_target.cax.set_yticklabels(['0% (Lost)', '50% (Truncated)', '100% (Intact)'], fontsize=10)
        g_target.cax.set_ylabel('Domain Completeness', rotation=270, labelpad=15, fontsize=12)
        
        plt.savefig('./Figures/Isoform Analysis/Isofrom_HM_Transposed.png', dpi=600, bbox_inches='tight')
        plt.close()
        print(f"Targeted Transposed Fractional Heatmap generated for {len(contrast_isoforms)} isoforms!")
else:
    print("No functional loss contrasts detected.")

Targeted Transposed Fractional Heatmap generated for 21 isoforms!


<Figure size 1260x600 with 0 Axes>